*** 1. Problem Definition ***

We aim to predict the wildfire events likelihood that the fire comes within 5 km of an evacuation zone at different future time horizon: [prob_12h prob_24h prob_48h prob_72h]


Target Y:  
* time_to_hit_hours = time until the wildfire reaches the zone
* event(wildfire) = 1 if it reached, 0 if censored

Feature X: test and train data columns


Our goal is to imporve the metrics:  
* HybridScore = 0.3 × C-inde x+ 0.7 × (1 − Weighted Brier Score)

In [ ]:
import pandas as pd
import numpy as np

# Load Data
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

# check data stucture
print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.head()


In [ ]:
# Check missing vals
missing_train = train.isnull().sum().sum()
missing_test = test.isnull().sum().sum()

print("Total missing values in train:", missing_train)
print("Total missing values in test:", missing_test)

In [ ]:
# targets
Y_time = train["time_to_hit_hours"]
Y_event = train["event"]

# Feature columns (exclude ID and target variables)
FEATURE_COLS = [
    col for col in train.columns
    if col not in ["event_id", "time_to_hit_hours", "event"]
]

X = train[FEATURE_COLS]
X_test = test[FEATURE_COLS]

print("Total number of features:", len(FEATURE_COLS))

In [ ]:
train.shape
train.head()

In [ ]:
# Visualize 
import matplotlib.pyplot as plt

train["time_to_hit_hours"].hist(bins=30)
plt.title("Distribution of Time to Hit (Hours)")
plt.xlabel("Hours")
plt.ylabel("Frequency")
plt.show()


### 2. DATA UNDERSTANDING AND AUDIT



In [ ]:
# some basic stats on the datasets
# Event distribution
print(f"\nEvent distribution:")
event_counts = train["event"].value_counts().sort_index()
print(f"  Censored (0): {event_counts.get(0, 0)} ({event_counts.get(0, 0)/len(train)*100:.1f}%)")
print(f"  Hit (1): {event_counts.get(1, 0)} ({event_counts.get(1, 0)/len(train)*100:.1f}%)")

# Time to hit stats (for events that hit)
hits = train[train["event"] == 1]
print(f"\nTime-to-hit distribution (hits only, n={len(hits)}):")
print(f"  Mean: {hits['time_to_hit_hours'].mean():.1f}h")
print(f"  Median: {hits['time_to_hit_hours'].median():.1f}h")
print(f"  Min: {hits['time_to_hit_hours'].min():.1f}h")
print(f"  Max: {hits['time_to_hit_hours'].max():.1f}h")
print(f"  25th: {hits['time_to_hit_hours'].quantile(0.25):.1f}h")
print(f"  75th: {hits['time_to_hit_hours'].quantile(0.75):.1f}h")

# Horizon feasibility
print(f"\nHits by horizon (feasibility check):")
for h in [12, 24, 48, 72]:
    n_hits = ((train["time_to_hit_hours"] <= h) & (train["event"] == 1)).sum()
    print(f"  ≤{h:2d}h: {n_hits:3d} hits ({n_hits/len(train)*100:.1f}% of dataset)")


In [ ]:
# Missing values audit
FEATURE_COLS = [col for col in train.columns 
                if col not in ["event_id", "time_to_hit_hours", "event"]]

missing_train = train[FEATURE_COLS].isnull().sum()
missing_test = test[FEATURE_COLS].isnull().sum()

if missing_train.sum() == 0:
    print("\nNo missing values in training features.")
else:
    print("\nMissing values in training features:")
    print(missing_train[missing_train > 0].sort_values(ascending=False))

if missing_test.sum() == 0:
    print("\nNo missing values in test features.")
else:
    print("\nMissing values in test features:")
    print(missing_test[missing_test > 0].sort_values(ascending=False))
    
# check for duplicates
n_dup = train.duplicated(subset=FEATURE_COLS).sum()
print(f"\nNumber of duplicate rows in training data: {n_dup}")

In [ ]:
# data leakage check, features that are highly correlated with the target
# Correlation with target time (suspicious if too high)
print("\nTop 10 features correlated with time_to_hit:")
correlations = train[FEATURE_COLS].corrwith(train["time_to_hit_hours"]).abs()
top_corr = correlations.sort_values(ascending=False).head(10)
for col, corr in top_corr.items():
    flag = "⚠️ " if corr > 0.7 else "  "
    print(f"{flag} {col:40s}: {corr:.3f}")

# Perfect separation check
print("\nChecking for perfect event separation:")
n_perfect = 0
for col in FEATURE_COLS:
    hit_min = train[train["event"] == 1][col].min()
    cens_max = train[train["event"] == 0][col].max()
    if hit_min > cens_max:
        print(f"{col}: Hits always > Censored")
        n_perfect += 1
        
if n_perfect == 0:
    print("No perfect separation found")

In [ ]:
# outliers - flag extreme outliers (5 IQR rule)
outlier_features = []
for col in FEATURE_COLS:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 5 * IQR
    upper = Q3 + 5 * IQR
    
    n_outliers = ((train[col] < lower) | (train[col] > upper)).sum()
    pct_outliers = n_outliers / len(train) * 100
    
    if pct_outliers > 5:  # Only flag if >5% are outliers
        outlier_features.append((col, n_outliers, pct_outliers))

if outlier_features:
    print(f"\nFeatures with >5% extreme outliers:")
    for col, n, pct in sorted(outlier_features, key=lambda x: x[2], reverse=True)[:10]:
        print(f"   {col:40s}: {n:3d} ({pct:5.1f}%)")
else:
    print("\nNo features with >5% extreme outliers")


### 3. DATA PREP:

In [ ]:
# Create binary labels for different horizons:
HORIZONS = [12, 24, 48, 72]

for horizon in HORIZONS:
    # Binary label: fire hit within this time window
    # Only label as 1 if event occurred AND time <= horizon
    train[f"hit_by_{horizon}h"] = (
        (train["event"] == 1) & (train["time_to_hit_hours"] <= horizon)
    ).astype(int)
    
    n_hits = train[f"hit_by_{horizon}h"].sum()
    print(f"\nHorizon {horizon:2d}h: {n_hits:3d} positives ({n_hits/len(train)*100:.1f}%)")
    
    FEATURE_COLS = [col for col in train.columns
                    if col not in ["event_id", "time_to_hit_hours", "event"] +
                    [f"hit_by_{h}h" for h in HORIZONS]]
    
    
print(f"\n Final feature count: {len(FEATURE_COLS)}")


### 4. CV setup

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# using stratified for inbalanced dataset

# CV config
n_splits = 5
n_repeats = 1 # increase to 3 after baseline works 
random_state = 42

# Stratify on event label to maintain hit/censor balance
stratify_col = train["event"]

# Initialize CV splitter
cv = StratifiedKFold(n_splits=n_splits, 
                     shuffle=True, 
                     random_state=random_state
                     )

print(f"\nStrategy: {n_splits}-Fold Stratified (on 'event')")
print(f"Repeats: {n_repeats} (for stability testing)")
print(f"\nFold distribution:")

# Show fold sizes and event distribution in each fold:
for fold, (train_idx, val_idx) in enumerate(cv.split(train, stratify_col)):
    train_events = train.iloc[train_idx]["event"].sum()
    val_events = train.iloc[val_idx]["event"].sum()
    
    print(f"  Fold {fold+1}: Train={len(train_idx)} ({train_events} hits), "
          f"Val={len(val_idx)} ({val_events} hits)")




### 5. Baseline modeling: